# Notebook 04: Efficiency → CREP Inversion (Γ_jet Calibration)

**GenesisAeon Package 17** — the central scientific derivation.

This notebook inverts the UTAC fixed-point relation to recover Γ_jet from
the measured η = 10% accretion-to-jet efficiency (Prabu et al. 2026).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from cygnus_jet_utac.efficiency import calibrate_gamma_jet, efficiency_from_gamma, gamma_scan
from cygnus_jet_utac.constants import UTAC_SIGMA_DEFAULT, UTAC_R_DEFAULT, SIGMA_PHI

np.random.seed(42)

## 1. The Inversion Formula

UTAC fixed point: $H^* = K \cdot \tanh(\sigma \cdot \Gamma)$

Efficiency: $\eta = H^*/K$

**Inversion:** $\Gamma_\text{jet} = \frac{\text{arctanh}(\eta)}{\sigma}$

In [ ]:
result = calibrate_gamma_jet(eta=0.10, verbose=True)
print(result['report'])

## 2. Sensitivity Analysis: η vs Γ_jet

In [ ]:
eta_arr, gamma_arr = gamma_scan()

# Frame Principle boundaries
sigma_phi_arr = UTAC_R_DEFAULT * (1 - np.tanh(UTAC_SIGMA_DEFAULT * gamma_arr)) / 2
frame_satisfied = sigma_phi_arr <= SIGMA_PHI

fig = plt.figure(figsize=(14, 5))
gs = gridspec.GridSpec(1, 3, figure=fig)

# Panel 1: η → Γ_jet
ax1 = fig.add_subplot(gs[0])
ax1.plot(eta_arr, gamma_arr, 'b-', lw=2)
ax1.axvline(0.10, color='red', ls='--', lw=1.5, label='η = 10%')
ax1.axhline(result['gamma_jet'], color='orange', ls='--', lw=1.5,
            label=f'Γ_jet = {result["gamma_jet"]:.4f}')
ax1.scatter([0.10], [result['gamma_jet']], color='red', s=100, zorder=5)
ax1.set_xlabel('Efficiency η')
ax1.set_ylabel('Γ_jet')
ax1.set_title('UTAC Inversion: η → Γ_jet')
ax1.legend(fontsize=8)
ax1.grid(alpha=0.3)

# Panel 2: Frame Principle
ax2 = fig.add_subplot(gs[1])
ax2.plot(gamma_arr, sigma_phi_arr, 'g-', lw=2, label='σ_Φ,min')
ax2.axhline(SIGMA_PHI, color='black', ls='--', lw=1.5, label=f'1/16 = {SIGMA_PHI:.4f}')
ax2.axvline(result['gamma_jet'], color='red', ls='--', lw=1.5,
            label=f'Γ_jet = {result["gamma_jet"]:.4f}')
ax2.fill_between(gamma_arr, sigma_phi_arr, SIGMA_PHI,
                  where=frame_satisfied, alpha=0.2, color='green', label='Frame OK')
ax2.set_xlabel('Γ_jet')
ax2.set_ylabel('σ_Φ,min')
ax2.set_title('Frame Principle Validation')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

# Panel 3: Across different σ values
ax3 = fig.add_subplot(gs[2])
for sigma in [1.5, 2.0, 2.2, 2.5, 3.0]:
    gamma_sigma = np.arctanh(eta_arr) / sigma
    ax3.plot(eta_arr, gamma_sigma, lw=1.5, label=f'σ={sigma}')
ax3.axvline(0.10, color='red', ls='--', lw=1.5, alpha=0.7)
ax3.set_xlabel('Efficiency η')
ax3.set_ylabel('Γ_jet')
ax3.set_title('Sensitivity to σ')
ax3.legend(fontsize=8)
ax3.grid(alpha=0.3)

fig.suptitle('Γ_jet Calibration — GenesisAeon Package 17', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/gamma_jet_calibration.png', dpi=150)
plt.show()
print('Saved: docs/gamma_jet_calibration.png')

## 3. Summary Table

In [ ]:
print('=== Key Results ===')
print(f'  Γ_jet           = {result["gamma_jet"]:.6f}')
print(f'  σ_Φ,min         = {result["sigma_phi_min"]:.6f}')
print(f'  σ_Φ,min/(1/16)  = {result["sigma_phi_ratio"]:.4f}')
print(f'  Frame Principle = {"✓ Satisfied" if result["sigma_phi_frame_principle_satisfied"] else "✗ Violated"}')
print(f'  H*/K check      = {result["utac_fixed_point_check"]:.6f} (should be {result["input_eta"]})')
print()
print(result['interpretation'])